# Phase B Backtest â€” PIT Universe + mean_reversion killed

Runs `run_backtest_production.py` on Kaggle with:
- **PIT_UNIVERSE_ENABLED = True** (797 historical NIFTY500 constituents)
- **mean_reversion signal killed** (Sharpe=-0.039, harmful)
- **11 active signals** with sector_rotation at 3%
- Period: 2012-01-01 to 2025-12-31 (~3190 trading days)

Baseline to beat: Sharpe=1.127, CAGR=30.4%, MaxDD=29.6%

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "yfinance", "requests", "beautifulsoup4", "lxml",
    "scikit-learn", "scipy", "ta", "arch"])
print("Dependencies installed.")

In [ ]:
import shutil, os, sys, glob

# Load code from the CODE DATASET (centurion-core).
# Kaggle mounts datasets at EITHER:
#   /kaggle/input/<slug>/                          (standard)
#   /kaggle/input/datasets/<owner>/<slug>/          (nested — seen in practice)
# And the zip may extract as:
#   .../centurion_core/services/   (with subdir wrapper)
#   .../services/                  (flat, no wrapper)
_CODE_CANDIDATES = [
    "/kaggle/input/centurion-core/centurion_core",
    "/kaggle/input/centurion-core",
    "/kaggle/input/datasets/srees16/centurion-core/centurion_core",
    "/kaggle/input/datasets/srees16/centurion-core",
]
src = None
for _candidate in _CODE_CANDIDATES:
    if os.path.isdir(_candidate) and os.path.isdir(os.path.join(_candidate, "services")):
        src = _candidate
        break

# Last-resort fallback: walk /kaggle/input
if src is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "services" in dirs and ("cloud" in dirs or "runners" in dirs):
            src = root
            break
        if "centurion_core" in dirs:
            candidate = os.path.join(root, "centurion_core")
            if os.path.isdir(os.path.join(candidate, "services")):
                src = candidate
                break

if src is None:
    import subprocess
    subprocess.run(["find", "/kaggle/input", "-maxdepth", "6", "-type", "d"], timeout=10)
    raise FileNotFoundError("centurion_core/ not found under /kaggle/input")

print(f"Found source: {src}")
dst = "/kaggle/working/centurion_core"
shutil.copytree(src, dst, dirs_exist_ok=True)
print(f"Copied {src} -> {dst}")

# ── Checkpoint restore: resume from previous Kaggle run if available ──
# On Kaggle, previous kernel output is at /kaggle/input/centurion-backtest-phase-b/
# (requires adding this kernel's own output as a data source in kernel settings)
_ckpt_dst = "/kaggle/working/centurion_core/data/backtest_checkpoint.pkl"
os.makedirs(os.path.dirname(_ckpt_dst), exist_ok=True)
_PREV_OUTPUT_CANDIDATES = [
    "/kaggle/input/centurion-backtest-phase-b/centurion_core/data/backtest_checkpoint.pkl",
    "/kaggle/input/centurion-backtest-phase-b/backtest_checkpoint.pkl",
]
_restored = False
for _ckpt_src in _PREV_OUTPUT_CANDIDATES:
    if os.path.exists(_ckpt_src):
        shutil.copy2(_ckpt_src, _ckpt_dst)
        _sz_mb = os.path.getsize(_ckpt_dst) / 1e6
        print(f"Restored checkpoint ({_sz_mb:.1f} MB) from {_ckpt_src}")
        _restored = True
        break
if not _restored:
    print("No previous checkpoint found \u2014 starting fresh run")

sys.path.insert(0, "/kaggle/working")
os.chdir("/kaggle/working")
print("Ready.")

In [ ]:
import subprocess, sys, time, os
print("=" * 70)
print("  Phase B Backtest — PIT Universe + mean_reversion killed")
print("  Universe: ~797 historical NIFTY500 constituents (PIT union)")
print("  Signals: 11 active (mean_reversion killed)")
print("  Period: 2012-01-01 to 2025-12-31")
print("=" * 70)

t0 = time.time()
env = {**os.environ, "CENTURION_NO_CHECKPOINT": "0", "CENTURION_MAX_RUNTIME_SECS": "42000"}  # checkpoints ON + 11h40m graceful exit
proc = subprocess.Popen(
    [sys.executable, "centurion_core/run_backtest_production.py"],
    cwd="/kaggle/working",
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
elapsed = (time.time() - t0) / 3600
print(f"\nBacktest finished in {elapsed:.1f}h - exit code {rc}")